In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

In [ ]:
df = pd.read_csv("../data/student-mat.csv", sep = ";")
df.head()

In [ ]:
df["G3"].describe()

In [ ]:
plt.figure(figsize = (7, 5))
plt.hist(df["G3"], bins = 10)
plt.title("Distribution of Final Grades (G3)")
plt.xlabel("Final Grade (G3)")
plt.ylabel("Number of Students")
plt.show()

In [ ]:
df.select_dtypes(include = "str").columns

In [ ]:
df_encoded = df.copy()

In [ ]:
from sklearn.preprocessing import LabelEncoder
le = LabelEncoder()
for column in df_encoded.select_dtypes(include='str').columns:
    df_encoded[column] = le.fit_transform(df_encoded[column])

In [ ]:
df_encoded.dtypes

In [ ]:
y = df_encoded["G3"]
X = df_encoded.drop("G3", axis = 1)
y.shape

In [ ]:
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size = 0.2,
    random_state = 42
)

In [ ]:
y_train.shape

In [ ]:
from sklearn.preprocessing import StandardScaler
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

In [ ]:
X_train_scaled[:5]

In [ ]:
from sklearn.linear_model import LinearRegression
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score
)

In [ ]:
lr = LinearRegression()
lr.fit(X_train_scaled, y_train)
lr_predictions = lr.predict(X_test_scaled)
lr_predictions[:10]

In [ ]:
comparison = pd.DataFrame({
    "Actual": y_test.values,
    "Predicted": lr_predictions
})
comparison.head(10)

In [ ]:
mae = mean_absolute_error(y_test, lr_predictions)
print("MAE", mae)

In [ ]:
mse = mean_squared_error(y_test, lr_predictions)
print("MSE", mse)

In [ ]:
import numpy as np

In [ ]:
rmse = np.sqrt(mse)
print("RMSE", rmse)

In [ ]:
r2 = r2_score(y_test, lr_predictions)
print("R2 Score", r2)

In [ ]:
dt = DecisionTreeRegressor(random_state = 42)
dt.fit(X_train_scaled, y_train)
dt_predictions = dt.predict(X_test_scaled)
print("MAE:", mean_absolute_error(y_test, dt_predictions))
print("MSE:", mean_squared_error(y_test, dt_predictions))
print("RMSE:", np.sqrt(mean_squared_error(y_test, dt_predictions)))
print("R2:", r2_score(y_test, dt_predictions))

In [ ]:
rf = RandomForestRegressor(random_state = 42)
rf.fit(X_train_scaled, y_train)
rf_predictions = rf.predict(X_test_scaled)
print("MAE:", mean_absolute_error(y_test, dt_predictions))
print("MSE:", mean_squared_error(y_test, dt_predictions))
print("RMSE:", np.sqrt(mean_squared_error(y_test, dt_predictions)))
print("R2:", r2_score(y_test, dt_predictions))

In [ ]:
results = pd.DataFrame({
    "Model": [
        "Linear Regression",
        "Decision Tree",
        "Random Forest"
    ],
    "MAE": [
        mean_absolute_error(y_test, lr_predictions),
        mean_absolute_error(y_test, dt_predictions),
        mean_absolute_error(y_test, rf_predictions)
    ],
    "RMSE": [
        np.sqrt(mean_squared_error(y_test, lr_predictions)),
        np.sqrt(mean_squared_error(y_test, dt_predictions)),
        np.sqrt(mean_squared_error(y_test, rf_predictions))
    ],
    "R2 Score": [
        r2_score(y_test, lr_predictions),
        r2_score(y_test, dt_predictions),
        r2_score(y_test, rf_predictions)
    ]
})

results

In [ ]:
plt.figure(figsize=(6,6))

plt.scatter(y_test, rf_predictions)

plt.xlabel("Actual Grades")

plt.ylabel("Predicted Grades")

plt.title("Actual vs Predicted Grades")

plt.show()

In [ ]:
importance = pd.DataFrame({
    "Feature": X.columns,
    "Importance": rf.feature_importances_
})

importance = importance.sort_values(
    by="Importance",
    ascending=False
)

importance.head(10)

In [109]:
rf_tuned = RandomForestRegressor(
    n_estimators=200,
    max_depth=15,
    random_state=42
)
rf_tuned.fit(X_train_scaled, y_train)
rf_tuned_predictions = rf_tuned.predict(X_test_scaled)
print("MAE:", mean_absolute_error(y_test, rf_tuned_predictions))
print("RMSE:", np.sqrt(mean_squared_error(y_test, rf_tuned_predictions)))
print("R2:", r2_score(y_test, rf_tuned_predictions))

MAE: 1.1208860759493673
RMSE: 1.9103990901233971
R2: 0.8220133494045573


In [99]:
from sklearn.model_selection import cross_val_score
scores = cross_val_score(
    rf_tuned,
    X,
    y,
    cv=5,
    scoring="r2"
)
print(scores.mean())

0.8352927292801198


In [105]:
import pickle
with open("../models/student_model.pkl", "wb") as file:
    pickle.dump(rf_tuned, file)

In [106]:
with open("../models/student_model.pkl", "rb") as file:
    loaded_model = pickle.load(file)

In [110]:
loaded_predictions = loaded_model.predict(X_test_scaled)
loaded_predictions[:5]

array([ 8.25 , 11.685,  6.78 ,  9.695,  9.095])

In [111]:
with open("../models/scaler.pkl", "wb") as file:
    pickle.dump(scaler, file)